# Assignment 2: Make prediction model interoperable 
### Group 4: Lisa van de Beek (i6261089), Daan Dieteren (i6275538), Casper Dijkstra (i6445875), Eva Dyadko (i6436149), Maggie Janisch (i6430641)
**Case:** Zhai et al. (2020)(https://doi.org/10.1016/j.radonc.2020.02.005)
**Date:** 3 April 2026

In [6]:
%%writefile myModel.py
from math import log, exp
from model_execution import cox_hazard_model

""" This code is used from Johan Soest's example code for logistic regression, which can be found here:
https://github.com/jvsoest/workshop_fairmodels

This code is adapted to fit the requirements of this assignment, and the model parameters are based on the paper by Zhai et al. (2020)(https://doi.org/10.1016/j.radonc.2020.02.005).

This is the standardized model code for the Zhai et al. (2020) model.
Coefficients and baseline hazard are based on the paper, Table 2
and the model_uri is based on the FHIR model registration.
The thresholds for the risk categories are based on the paper, found in the Abstract, and are as follows:
- 0.10 for low risk
- 0.60 for high risk
- anything in between is medium risk
"""

class Zhai_predictLymph(cox_hazard_model):
    def __init__(self):
        self._model_parameters = {
            "model_uri": "https://v3.fairmodels.org/instance/8da56fed-5f87-4699-9b69-38b255b08268",
            "model_name": "Pre treatment radiomic features predict individual lymph node failure for head and neck cancer patients.",
            "baseline_hazard": 0.000528258, # 2 year baseline hazard based on appendix C of paper
            "covariate_weights": { # can be found in Table 2 of the paper
                "Gender": 2.19,     
                "T_stage": 1.08,    
                "WHO_PS": 1.06,     
                "LALLN": 0.83,        
                "Corr_GLCM": 2.84
            }
        }
    def _preprocess(self, data):
        """
        This function preprocesses the input data using standardized SNOMED and LOINC codes, based on our report
        
        Parameters:
            - data (dict): Patient JSON file containing Patient, Condition, and Observation resources.

        Returns:
            - dict: Preprocessed data : Numeric values for Gender, T_stage, WHO_PS, LALLN, Corr_GLCM.

        """

        extracted_data = {
            "Gender": None,
            "T_stage": None,
            "WHO_PS": None,
            "LALLN": None,
            "Corr_GLCM": None
        }

        # Get the relevant data from the input JSON, using the standardized codes
        entries = data.get('entry', [])
        for entry in entries:
            resource = entry.get('resource', {})
            resource_type = resource.get('resourceType')

            # Gender check for SNOMED codes for gender:
            # Male: 248153007 and Female: 248152002
            if resource_type == 'Patient':
                for extension in resource.get('extension', []):
                    for coding in extension.get('valueCodeableConcept', {}).get('coding', []):
                        code = coding.get('code')
                        if code == '248153007': # male code
                            extracted_data['Gender'] = 1.0
                        elif code == '248152002': # female code
                            extracted_data['Gender'] = 0.0

            # T-Stage: Check for SNOMED T3 (1228889006) or T4 (1228888003)
            # T1 and T0 stay at 0.0, which is the default value
            if resource_type == 'Condition':
                for stage in resource.get('stage', []):
                    for coding in stage.get('summary', {}).get('coding', []):
                        t_code = coding.get('code')
                        if t_code in ['1228889006', '1228888003']: # T3 or T4 code
                            extracted_data['T_stage'] = 1.0
                        elif t_code in ['1228887008', '1228886004']: # T1 or T2 code
                            extracted_data['T_stage'] = 0.0

            #  WHO_PS, LALLN, and Corr_GLCM
            if resource_type == 'Observation':
                for coding in resource.get('code', {}).get('coding', []):
                    obs_code = coding.get('code')

                    # WHO Performance Status (LOINC: C1298651) check for graades 1-4
                    if obs_code == 'C1298651':
                        for value in resource.get('valueCodeableConcept', {}).get('coding', []):
                            v_code = value.get('code')
                            if v_code == '373797003': # code for WHO PS 0
                                extracted_data['WHO_PS'] = 0.0
                            elif v_code in ['373798008', '373799000', '373800008', '373801007']: # codes for WHO PS 1-4
                                extracted_data['WHO_PS'] = 1.0

                    # LALLN
                    if obs_code == 'least-axis-length-llln':
                        val = resource.get('valueQuantity', {}).get('value')
                        if val is not None:
                            extracted_data['LALLN'] = float(val)

                    # Corr_GLCM
                    if obs_code == 'correlation-glcm':
                        val = resource.get('valueDecimal')
                        if val is not None:
                            extracted_data['Corr_GLCM'] = float(val)
                    
                    
        # Handle missing values
        missing_features = []
        for key in extracted_data:
            if extracted_data[key] is None:
                missing_features.append(key)

        if len(missing_features) > 0:
            raise ValueError(f"Missing values for features: {', '.join(missing_features)}")
    
        return extracted_data


    def predict(self, data):
        """
        This function takes the preprocessed data and calculates the risk score based on the model parameters,
        for 2 year nodal failure risk.

        Parameters:
            data (dict): Preprocessed data from the _preprocess function.
 
        Returns:
            dict: A dictionary containing:
                - model_uri (str): URI of the model.
                - risk_score (str): Calculated risk score as a percentage string.
                - risk_category (str): Risk classification (High risk, Medium risk, Low risk).
        """
        
        # Error handling for missing values in the preprocessed data
        try:
            preprocessed_data = self._preprocess(data)
        except ValueError as ve:
            return {
                "model_uri": self._model_parameters["model_uri"],
                "error": f"Incomplete data: {ve}"
            }

        weights = self._model_parameters['covariate_weights']

        # Calculate the linear predictor
        linear_predictor = 0.0
        for key in weights:
            linear_predictor = linear_predictor + (preprocessed_data[key] * weights[key])

        # Calculate the risk score using the Cox model formula
        risk_score = 1 - exp(-self._model_parameters['baseline_hazard'] * exp(linear_predictor))

        # Classify the risk category based on 60% and 10% thresholds
        if risk_score >= 0.60:
            risk_category = "High risk"
        elif risk_score <= 0.10:
            risk_category = "Low risk"
        else:
            risk_category = "Medium risk"

        return {
            "model_uri": self._model_parameters["model_uri"],
            "risk_score": f"{round(risk_score * 100, 2)}%",
            "risk_category": risk_category
        }



Overwriting myModel.py
